In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import stats
import math
import glob
import pandas as pd

In [ ]:
#Only noise (\eta)segmentation in albedo images obtained with Bell method

from scipy import stats

folder = "albedo_Bell"
files = glob.glob(folder+"/*.png")
files.sort()
print (files)

coef_S = 1.3
coef_H = 1



df = pd.DataFrame() 
idx = 0

for f in files:
    
    print (f)
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
    
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)

        h_min = np.min(h_roi)
        h_max = np.max(h_roi)
        print (h_min,   h_max)

        if h_min<0 and h_max>200:
            for i in h_roi:
                if i<0:
                    #index = h_roi.index(i)
                    #print (index)
                    j = 360 + i
                    print (j)
                    print (h_roi[index])
                    h_roi[index] = int(j)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)
        
        h = np.mean(h_no_out)
        h_min = np.min(h_no_out)
        h_max = np.max(h_no_out)
        h_noise = h_max-h_min
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("H: ", h)
        

        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)

        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s 
        print ("S: ", s)


        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        v_max = np.max(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
        

        rang_H = -0.08*s + 6.88  #Són les equacions de la 90D
        rang_S = 0.12*s + 0.56 
        
        print ("Rang H eq: ", rang_H)
        print ("H noise: ", h_noise)
        print ("Rang S eq: ", rang_S)
        print ("S noise: ", s_noise)

    
        c1 = (h_min, min_s, 0) #per sota només resto 1/3 part del rang_S (perquè no estic mesurant a la zona més lluminosa) i el noise perquè rang S és assimètric
        c2 = (h_max, max_s, v_max)
    
        print(c1,c2)

        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-7:-4]+"_Bell_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-7:-4]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "H"] = h
        df.at[idx, "S"] = s
        df.at[idx, "V"] = v
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "S noise"] = s_noise
        df.at[idx, "coef H"] = coef_H
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range apl"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef S"] = coef_S
        df.at[idx, "S range eq"] = rang_S
        df.at[idx, "S range apl"] = (rang_S+s_noise/2)*coef_S+rang_S/3+s_noise/2

        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Bell_results.xlsx") 

In [ ]:
#Only noise (\eta)segmentation in albedo images obtained with Careaga method

from scipy import stats

folder = "albedo_Careaga"
files = glob.glob(folder+"/*.png")
files.sort()
print (files)

coef_S = 1.3
coef_H = 1



df = pd.DataFrame() 
idx = 0

for f in files:
    
    print (f)
    index = 1
    
    img = cv2.imread(f)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print (f)
        
    size = img.shape
    img_hsv = np.zeros((size[0], size[1], 3))
        
    for i in range (img.shape[0]):
        for j in range (img.shape[1]):
            R = int(img_rgb[i,j,0])
            G = int(img_rgb[i,j,1])
            B = int(img_rgb[i,j,2])
            
            RGB = (R,G,B)
            #print (RGB)
            
            maxi = max(RGB)
            mini = min(RGB)
            
            if maxi == mini:
                mini = maxi-1
            
            if maxi == RGB[0]:
                H = 60*((G-B)/(maxi-mini))
            elif maxi == RGB[1]:
                H = 120+60*((B-R)/(maxi-mini))
            elif maxi == RGB[2]:
                H = 240+60*((R-G)/(maxi-mini)) 
                
            if maxi == 0:
                S = 0
            else:
                #s = 100*((maxi-mini)/maxi) 
                S = 100*(float((maxi-mini)/maxi))
            
            V = (maxi/255)*100  
            
            img_hsv[i,j,2] = V
            img_hsv[i,j,1] = S
            img_hsv[i,j,0] = H

    if "287" in f:
        colors = ([302, 344], [626, 305], [493, 254], [507, 253], [627, 265])

    elif "307" in f:
        colors = ([160, 653], [252, 817], [220, 712], [219, 584], [138, 659], [302, 852], [165, 710],[242, 570])

    elif "305" in f:
        colors = ([466, 276], [400, 313], [519, 326], [1013, 521], [420, 367])  

    elif "291" in f:
        colors = ([526, 708], [531, 394], [420, 607], [380, 475], [450, 707])
    
    for c in colors:
                
        x = c[0]
        y = c[1]
        
        winsz = 8   #ajustar en f(x) del soroll/textura

        color_roi = (255,0,0)
        
        #img_roi = []
        
        img = cv2.imread(f)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_roi = cv2.rectangle(img_rgb, (y-(winsz//2), x-(winsz//2)), (y+winsz, x+winsz), color_roi, 3)

        h_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),0]
        #h_circ_np = (h_roi_np)*2
        
        h_roi = h_roi_np.flatten()
        h_mean = np.mean(h_roi)
        h_std = np.std(h_roi)
        #print (h_roi)

        h_min = np.min(h_roi)
        h_max = np.max(h_roi)
        print (h_min,   h_max)

        if h_min<0 and h_max>200:
            for i in h_roi:
                if i<0:
                    #index = h_roi.index(i)
                    #print (index)
                    j = 360 + i
                    print (j)
                    print (h_roi[index])
                    h_roi[index] = int(j)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((h_roi > (h_mean+k*h_std)) | (h_roi < (h_mean-k*h_std)))[0]
        print ("H outliers:",  outlier_indices)
        h_no_out = np.delete(h_roi, outlier_indices)
        
        h = np.mean(h_no_out)
        h_min = np.min(h_no_out)
        h_max = np.max(h_no_out)
        h_noise = h_max-h_min
        #std_h = stats.circstats.circstd(h_circ, method='angular')
        print ("H: ", h)
        

        s_roi_np = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),1]
        
       #Descartar outliers amb mètode Chevyshev
        s_roi = s_roi_np.flatten()
        s_mean = np.mean(s_roi)
        s_std = np.std(s_roi)
        #print (h_roi)
        
        k = 3.16  #equival a p = 0.1 
        outlier_indices = np.where((s_roi > (s_mean+k*s_std)) | (s_roi < (s_mean-k*s_std)))[0]
        print ("S outliers:",  outlier_indices)
        s_no_out = np.delete(s_roi, outlier_indices)

        s = np.mean(s_no_out)
        std_s = np.std(s_no_out)
        min_s = np.min(s_no_out)
        max_s = np.max(s_no_out)
        s_noise = max_s-min_s 
        print ("S: ", s)


        v_roi = img_hsv[(x-(winsz//2)):(x+(winsz//2)),(y-(winsz//2)):(y+(winsz//2)),2]
        v = np.mean(v_roi)
        v_max = np.max(v_roi)
        std_v = np.std(v_roi)
        print ("V: ", v)
        

        rang_H = -0.08*s + 6.88  #Són les equacions de la 90D
        rang_S = 0.12*s + 0.56 
        
        print ("Rang H eq: ", rang_H)
        print ("H noise: ", h_noise)
        print ("Rang S eq: ", rang_S)
        print ("S noise: ", s_noise)

    
        c1 = (h_min, min_s, 0) #per sota només resto 1/3 part del rang_S (perquè no estic mesurant a la zona més lluminosa) i el noise perquè rang S és assimètric
        c2 = (h_max, max_s, v_max)
    
        print(c1,c2)

        mask = cv2.inRange(img_hsv, c1, c2)
        result = cv2.bitwise_and(img, img, mask=mask)
        result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
        result_name = "../results/"+f[-7:-4]+"_Careaga_"+str(winsz)+"_"+str(index)+".png"
        cv2.imwrite (result_name, result)


        img_roi_rgb = cv2.cvtColor(img_roi, cv2.COLOR_RGB2BGR)
        img_roi_name = "../results/"+f[-7:-4]+"ROI"+str(index)+".png"
        cv2.imwrite (img_roi_name, img_roi_rgb)

        df.at[idx, "Image"] = f
        df.at[idx, "color"] = str(index)
        df.at[idx, "H"] = h
        df.at[idx, "S"] = s
        df.at[idx, "V"] = v
        df.at[idx, "H noise"] = h_noise
        df.at[idx, "S noise"] = s_noise
        df.at[idx, "coef H"] = coef_H
        df.at[idx, "H range eq"] = rang_H
        df.at[idx, "H range apl"] = (rang_H+h_noise)*coef_H
        df.at[idx, "Coef S"] = coef_S
        df.at[idx, "S range eq"] = rang_S
        df.at[idx, "S range apl"] = (rang_S+s_noise/2)*coef_S+rang_S/3+s_noise/2

        idx += 1
        
        fig = plt.figure(figsize = (40,15))
        plt.subplot(1, 2, 1)
        plt.imshow(img_roi)
        plt.subplot(1,2,2)
        plt.imshow(result_bgr)
        plt.show()
        
    
        index += 1

print(df)
df.to_excel("../results/Careaga_results.xlsx") 